# D3 model evaluation: baseline vs JobSwipe vs Job-Tinder
**DTSC3602 Group 9 · "Tinder for Jobs"**

This notebook compares a baseline and the group's two candidate fit models, using one validation protocol and one set of metrics.

| Role | Model | Source |
|---|---|---|
| Baseline | TF-IDF cosine (untrained) | this notebook |
| Candidate A | JobSwipe: gte-small profile ↔ job cosine | github.com/Baguirr1/jobswipe @ `fa50c8b` |
| Candidate B | Job-Tinder: LightGBM on 11 pairwise features | github.com/awalla63/Job-Tinder-Test-build @ `cef3a70` |

- **Data:** `cnamuangtoun/resume-job-description-fit`, 8,000 labelled pairs (No / Potential / Good Fit).
- **Splits:** cold-JD (the dataset's own split) and cold-resume (5-fold GroupKFold by resume).
- **Metrics:** AUC Good-vs-No with cluster-bootstrap CIs, macro F1, NDCG@10 (seeker and employer decks), and the share of No Fit cards in each seeker's top 5.

Run it from the repo root. The first run takes ~10 min on CPU (embedding); later runs take ~2–3 min because the embeddings are cached in `eval/cache/`.

## 0. Setup (run once)
Clone both candidate repos at the evaluated commits and install the pinned dependencies. Node ≥ 22.18 is also needed, to run JobSwipe's own TypeScript.

In [ ]:
# Uncomment on first run.
# !git clone https://github.com/Baguirr1/jobswipe repos/jobswipe
# !git clone https://github.com/awalla63/Job-Tinder-Test-build repos/job_tinder_test
# !git -C repos/jobswipe checkout fa50c8b
# !git -C repos/job_tinder_test checkout cef3a70
# %pip install -r requirements.txt

## 1. JobSwipe text builder (TypeScript)
This builds the exact strings JobSwipe embeds by importing JobSwipe's own `embeddings.ts` and `resumeExtract.ts` unchanged. The cell writes the file, and Python calls it with `node` in step 3.

In [ ]:
%%writefile eval/jobswipe_texts.mts
// Build the exact strings JobSwipe embeds, using JobSwipe's own TypeScript.
//
// Nothing here re-implements JobSwipe logic: buildProfileTexts / buildJobText /
// stripContactInfo / chunkText come from supabase/functions/_shared/embeddings.ts and
// extractFromResume from src/lib/resumeExtract.ts, imported unchanged from the cloned
// repo. Node >= 22.18 strips the TypeScript types natively.
//
//   node eval/jobswipe_texts.mts <in.json> <out.json>
//
// in.json:  {"resumes": [str, ...], "jobs": [str, ...]}
// out.json: {"profiles_suggested": [{header, chunks}], "profiles_resume_only": [...],
//            "jobs": [str, ...]}
import { readFileSync, writeFileSync } from 'node:fs';
import { buildJobText, buildProfileTexts } from '../repos/jobswipe/supabase/functions/_shared/embeddings.ts';
import { extractFromResume } from '../repos/jobswipe/src/lib/resumeExtract.ts';

const [inPath, outPath] = process.argv.slice(2);
type JobIn = string | { title: string; description: string; location?: string | null; is_remote?: boolean | null };
const { resumes, jobs } = JSON.parse(readFileSync(inPath, 'utf8')) as { resumes: string[]; jobs: JobIn[] };

// "Suggested" profile: the onboarding flow where the seeker uploads a resume and keeps
// every suggestion the "From your resume" card offers (title, skills, summary -> bio).
const profiles_suggested = resumes.map((text) => {
  const s = extractFromResume(text);
  return buildProfileTexts({ target_title: s.title, skills: s.skills, bio: s.summary, resume_text: text });
});

// Resume-only profile: the seeker uploads a resume and accepts nothing else.
const profiles_resume_only = resumes.map((text) => buildProfileTexts({ resume_text: text }));

// The labelled pairs carry only a description (a plain string): no title / skills / location,
// so the empty "Role: " prefix is dropped. Real postings (objects) get JobSwipe's full header.
const jobTexts = jobs.map((j) =>
  typeof j === 'string' ? buildJobText({ title: '', description: j }).replace(/^Role: /, '') : buildJobText(j),
);

writeFileSync(outPath, JSON.stringify({ profiles_suggested, profiles_resume_only, jobs: jobTexts }));
console.log(`built ${resumes.length} profiles and ${jobs.length} job texts`);

## 2. Imports and configuration

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()  # run this notebook from the repo root
JOBTINDER = ROOT / "repos" / "job_tinder_test"
sys.path.insert(0, str(JOBTINDER))

import lightgbm as lgb  # noqa: E402
from sklearn.metrics import confusion_matrix, f1_score  # noqa: E402
from sklearn.model_selection import GroupKFold  # noqa: E402

from ml.evaluate import (  # noqa: E402  Job-Tinder's metric code, reused unchanged
    auc_good_vs_no,
    build_features,
    cluster_bootstrap_auc,
    group_columns,
    metrics as base_metrics,
    paired_auc_gap_ci,
)
from ml.features import ALL_FEATURE_NAMES, FEATURE_SETS, LABEL_TO_INT, LABELS, fit_score  # noqa: E402
from ml.train import ensure_dataset, make_params  # noqa: E402

OUT = ROOT / "results"
CACHE = ROOT / "eval" / "cache"
GOOD, POT, NO = LABEL_TO_INT["Good Fit"], LABEL_TO_INT["Potential Fit"], LABEL_TO_INT["No Fit"]

BASELINE = "tfidf_cosine"
CANDIDATES = ["jobswipe_gte_cosine", "jobtinder_lightgbm"]
SHOWN = ["random", BASELINE, *CANDIDATES]  # rows in the error analysis

# JobSwipe's display mapping (supabase/migrations/..._gte_small_embeddings.sql, match_pct)
MATCH_LO, MATCH_HI = 0.80, 0.96
JOB_CHAR_CAP = 1500  # JobSwipe's MAX_INPUT_CHARS
HEADER_WEIGHT = 0.5  # JobSwipe's HEADER_WEIGHT


# Run settings
ROUNDS = 400   # LightGBM boosting rounds (Job-Tinder's default)
SEED = 42
FOLDS = 5      # cold-resume GroupKFold folds
BOOT = 1000    # bootstrap resamples (200 is faster)

## 3. Candidate A: JobSwipe scores
This computes the gte-small cosine between JobSwipe's profile vector (header weighted 0.5, plus up to two resume chunks) and its job vector (the JD's first 1,500 chars).

In [ ]:
def jobswipe_scores(all_df: pd.DataFrame) -> dict[str, np.ndarray]:
    """gte-small cosine per pair, for the 'suggested' and 'resume-only' profiles."""
    CACHE.mkdir(parents=True, exist_ok=True)
    resumes = sorted(set(all_df.resume_text))
    jobs = sorted(set(all_df.job_description_text))

    texts_path = CACHE / "jobswipe_texts.json"
    if not texts_path.exists():
        src = CACHE / "jobswipe_input.json"
        src.write_text(json.dumps({"resumes": resumes, "jobs": jobs}))
        subprocess.run(["node", str(ROOT / "eval" / "jobswipe_texts.mts"), str(src), str(texts_path)],
                       check=True, cwd=ROOT)
    texts = json.loads(texts_path.read_text(encoding="utf-8"))

    emb_path = CACHE / "gte_small_embeddings.npz"
    flat = sorted({t for v in ("profiles_suggested", "profiles_resume_only")
                   for p in texts[v] for t in [p["header"], *p["chunks"]] if t} | set(texts["jobs"]))
    if emb_path.exists() and list(np.load(emb_path, allow_pickle=True)["keys"]) == flat:
        vecs = np.load(emb_path, allow_pickle=True)["vectors"]
    else:
        from sentence_transformers import SentenceTransformer
        # thenlper/gte-small is the model Supabase.ai serves as 'gte-small' (mean pooled,
        # L2-normalised, 512-token window).
        model = SentenceTransformer("thenlper/gte-small", device="cpu")
        model.max_seq_length = 512
        print(f"  embedding {len(flat):,} JobSwipe strings with gte-small ...", flush=True)
        vecs = model.encode(flat, batch_size=16, normalize_embeddings=True,
                            convert_to_numpy=True, show_progress_bar=True)
        np.savez_compressed(emb_path, keys=np.array(flat, dtype=object), vectors=vecs)
    lookup = dict(zip(flat, vecs))

    def profile_vec(p: dict) -> np.ndarray:
        # generateProfileEmbedding(): header 0.5 + chunks share 0.5, then re-normalise.
        h, chunks = p["header"], p["chunks"]
        if h and chunks:
            parts = [(h, HEADER_WEIGHT)] + [(c, (1 - HEADER_WEIGHT) / len(chunks)) for c in chunks]
        elif h:
            parts = [(h, 1.0)]
        else:
            parts = [(c, 1 / len(chunks)) for c in chunks]
        v = sum(w * lookup[t] for t, w in parts)
        return v / np.linalg.norm(v)

    job_vec = {j: lookup[t] for j, t in zip(jobs, texts["jobs"])}
    out = {}
    for variant, key in (("jobswipe_gte_cosine", "profiles_suggested"),
                         ("jobswipe_resume_only", "profiles_resume_only")):
        res_vec = {r: profile_vec(p) for r, p in zip(resumes, texts[key])}
        out[variant] = np.array([float(res_vec[r] @ job_vec[j]) for r, j in
                                 zip(all_df.resume_text, all_df.job_description_text)])
    out["_header_has_title"] = {r: bool(p["header"].startswith("Role:"))
                                for r, p in zip(resumes, texts["profiles_suggested"])}
    return out

## 4. Models
Untrained similarities (baseline, JobSwipe) rank by score and turn it into 3 classes at cut-points matching the training fold's label mix. Job-Tinder's LightGBM is retrained in every training fold with its own parameters.

In [ ]:
def prior_cuts(train_scores, y_train):
    shares = np.bincount(y_train, minlength=3) / len(y_train)
    return np.quantile(train_scores, np.cumsum(shares)[:2])


def score_model(scores: np.ndarray):
    """An untrained similarity: rank by the score, classify with training-mix cut-points."""
    def fit(tr, y):
        cuts = prior_cuts(scores[tr], y[tr])
        return lambda idx: (scores[idx], np.searchsorted(cuts, scores[idx], side="right"))
    return fit


def lightgbm_model(X: np.ndarray, rounds: int, seed_offset: list[int]):
    """Job-Tinder's shipped model: LightGBM on the pairwise_plus features, its params."""
    names = FEATURE_SETS["pairwise_plus"]
    cols = [ALL_FEATURE_NAMES.index(n) for n in names]

    def fit(tr, y):
        booster = lgb.train(make_params(seed_offset[0]),
                            lgb.Dataset(X[tr][:, cols], label=y[tr], feature_name=names),
                            num_boost_round=rounds)

        def predict(idx):
            p = booster.predict(X[idx][:, cols])
            return fit_score(p), p.argmax(axis=1)
        return predict
    return fit

## 5. Metrics and error-analysis helpers

In [ ]:
def nofit_top5(y, score, resume_ids) -> float:
    """Mean share of No Fit pairs among each resume's 5 highest-scored jobs (resumes with >=5)."""
    frame = pd.DataFrame({"g": resume_ids, "y": y, "s": score})
    vals = [(c.nlargest(5, "s").y == NO).mean() for _, c in frame.groupby("g") if len(c) >= 5]
    return round(float(np.mean(vals)), 4)


def all_metrics(y, score, pred, jd_ids, resume_ids) -> dict:
    r = base_metrics(y, score, pred, jd_ids, resume_ids)
    r["nofit_share_top5_seeker"] = nofit_top5(y, score, resume_ids)
    r["confusion"] = confusion_matrix(y, pred, labels=[0, 1, 2]).tolist()
    return r


def extra_groups(all_df: pd.DataFrame, X: np.ndarray, has_title: dict) -> dict[str, pd.Series]:
    """Groups specific to how the two candidates read text (truncation, title detection)."""
    jd_len = all_df.job_description_text.str.len()
    res_len = all_df.resume_text.str.len()
    groups = group_columns(X)
    groups["jd_length"] = pd.Series(np.where(jd_len <= JOB_CHAR_CAP, "<=1,500 chars (fully read)",
                                    np.where(jd_len <= 3000, "1,501-3,000", ">3,000")))
    groups["resume_length"] = pd.Series(pd.cut(res_len, [0, 3000, 6000, 1e9],
                                               labels=["<=3,000 chars", "3,001-6,000", ">6,000"])
                                        .astype(str).to_numpy())
    groups["resume_title_detected"] = pd.Series(
        ["title found" if has_title[r] else "no title found" for r in all_df.resume_text])
    return groups


def error_breakdown(groups, y, oof) -> dict:
    out = {}
    for condition, g in groups.items():
        rows = []
        for value in sorted(g.unique(), key=str):
            m = (g == value).to_numpy()
            row = {"group": str(value), "n": int(m.sum()),
                   "good_fit_share": round(float((y[m] == GOOD).mean()), 3)}
            for name, (score, pred) in oof.items():
                severe = ((y[m] == NO) & (pred[m] == GOOD)) | ((y[m] == GOOD) & (pred[m] == NO))
                row[name] = {"auc": round(auc_good_vs_no(y[m], score[m]), 4),
                             "macro_f1": round(float(f1_score(y[m], pred[m], average="macro")), 4),
                             "severe_error_rate": round(float(severe.mean()), 4)}
            rows.append(row)
        out[condition] = rows
    return out


def match_pct(cos: np.ndarray) -> np.ndarray:
    return np.clip((cos - MATCH_LO) / (MATCH_HI - MATCH_LO), 0, 1) * 100

## 6. Load data and build features
This step downloads the labelled dataset on first run, builds Job-Tinder's features (TF-IDF, bge-small, skills, years, seniority) and computes JobSwipe's scores.

In [ ]:
t0 = time.time()

train_df, test_df = ensure_dataset()
all_df = pd.concat([train_df, test_df], ignore_index=True)
y = all_df.label.map(LABEL_TO_INT).to_numpy()
is_test = np.r_[np.zeros(len(train_df), bool), np.ones(len(test_df), bool)]
jd_ids, res_ids = all_df.job_description_text.to_numpy(), all_df.resume_text.to_numpy()
print(f"{len(all_df):,} pairs | {len(set(res_ids))} resumes | {len(set(jd_ids))} JDs")

print("Job-Tinder features (TF-IDF, bge-small, skills, years, seniority) ...")
X = build_features(all_df)
print("JobSwipe gte-small scores ...")
js = jobswipe_scores(all_df)
rng = np.random.default_rng(SEED)

seed_box = [SEED]
models = {
    "random": ("reference", score_model(rng.random(len(y)))),
    "tfidf_cosine": ("baseline", score_model(X[:, ALL_FEATURE_NAMES.index("tfidf_cosine")])),
    "jobswipe_gte_cosine": ("candidate", score_model(js["jobswipe_gte_cosine"])),
    "jobtinder_lightgbm": ("candidate", lightgbm_model(X, ROUNDS, seed_box)),
    "jobswipe_resume_only": ("ablation", score_model(js["jobswipe_resume_only"])),
    "bge_cosine": ("ablation", score_model(X[:, ALL_FEATURE_NAMES.index("emb_cosine")])),
}

## 7. Cold-JD split
Train on the dataset's train split and test on unseen job descriptions.

In [ ]:
# --- Split 1: cold-JD -------------------------------------------------------
print("Cold-JD split ...")
tr, te = np.flatnonzero(~is_test), np.flatnonzero(is_test)
cold_jd, cj_scores = {}, {}
for name, (_, fit) in models.items():
    seed_box[0] = SEED
    score, pred = fit(tr, y)(te)
    r = all_metrics(y[te], score, pred, jd_ids[te], res_ids[te])
    r["auc_ci"] = cluster_bootstrap_auc(y[te], score, jd_ids[te], BOOT, SEED)
    cold_jd[name], cj_scores[name] = r, score

## 8. Cold-resume split
5-fold GroupKFold by resume, out-of-fold predictions for all 8,000 pairs.

In [ ]:
# --- Split 2: cold-resume ---------------------------------------------------
print(f"Cold-resume split ({FOLDS}-fold GroupKFold) ...")
folds = list(GroupKFold(n_splits=FOLDS).split(X, y, res_ids))
cold_res, oof = {}, {}
for name, (_, fit) in models.items():
    score, pred = np.zeros(len(y)), np.zeros(len(y), int)
    for k, (ftr, fva) in enumerate(folds, 1):
        seed_box[0] = SEED + k  # matches Job-Tinder's train.py fold seeds
        score[fva], pred[fva] = fit(ftr, y)(fva)
    r = all_metrics(y, score, pred, jd_ids, res_ids)
    r["auc_ci"] = cluster_bootstrap_auc(y, score, res_ids, BOOT, SEED)
    cold_res[name], oof[name] = r, (score, pred)

## 9. Paired AUC gaps
These are 95% cluster-bootstrap CIs on the difference between two models' AUCs, computed on the same resamples.

In [ ]:
# --- Paired comparisons -----------------------------------------------------
pairs = [("jobtinder_lightgbm", BASELINE), ("jobswipe_gte_cosine", BASELINE),
         ("jobtinder_lightgbm", "jobswipe_gte_cosine"),
         ("jobswipe_gte_cosine", "jobswipe_resume_only"), ("jobswipe_gte_cosine", "bge_cosine")]
gaps = {}
for a, b in pairs:
    gaps[f"{a} - {b}"] = {
        "cold_jd": {"point": round(cold_jd[a]["auc_good_vs_no"] - cold_jd[b]["auc_good_vs_no"], 4),
                    "ci": paired_auc_gap_ci(y[te], cj_scores[a], cj_scores[b], jd_ids[te],
                                            BOOT, SEED)},
        "cold_resume": {"point": round(cold_res[a]["auc_good_vs_no"] - cold_res[b]["auc_good_vs_no"], 4),
                        "ci": paired_auc_gap_ci(y, oof[a][0], oof[b][0], res_ids,
                                                BOOT, SEED)},
    }

## 10. Error analysis and saving results
Per-group AUC, macro F1 and severe-error rate on the cold-resume out-of-fold predictions. Also covers the most confident mistakes, JobSwipe's "% match" badge by true label, and writes `results/evaluation.json`.

In [ ]:
# --- Error analysis (cold-resume OOF: every pair scored by a model blind to its resume)
groups = extra_groups(all_df, X, js["_header_has_title"])
errors = error_breakdown(groups, y, {n: oof[n] for n in SHOWN})

# Most confidently wrong pairs for the two candidates (for the write-up)
worst = {}
for name in CANDIDATES:
    s, p = oof[name]
    pct = pd.Series(s).rank(pct=True).to_numpy()
    fp = np.flatnonzero((y == NO))[np.argsort(-pct[y == NO])[:5]]
    fn = np.flatnonzero((y == GOOD))[np.argsort(pct[y == GOOD])[:5]]
    worst[name] = {kind: [{"score_percentile": round(float(pct[i]), 3),
                           "resume_start": all_df.resume_text[i][:160],
                           "jd_start": all_df.job_description_text[i][:160]} for i in idx]
                   for kind, idx in (("no_fit_ranked_highest", fp), ("good_fit_ranked_lowest", fn))}

# JobSwipe's user-facing "% match" badge, by true label
mp = match_pct(js["jobswipe_gte_cosine"])
badge = {lab: {"mean_match_pct": round(float(mp[y == i].mean()), 1),
               "share_shown_>=70%": round(float((mp[y == i] >= 70).mean()), 3),
               "share_shown_<30%": round(float((mp[y == i] < 30).mean()), 3)}
         for i, lab in enumerate(LABELS)}
cos = js["jobswipe_gte_cosine"]
badge["raw_cosine_range"] = [round(float(cos.min()), 4), round(float(cos.max()), 4)]

report = {
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "duration_seconds": round(time.time() - t0, 1),
    "data": {"n_pairs": int(len(y)), "n_resumes": int(len(set(res_ids))),
             "n_jds": int(len(set(jd_ids))), "n_cold_jd_test": int(is_test.sum()),
             "label_share": {lab: round(float((y == i).mean()), 3) for i, lab in enumerate(LABELS)}},
    "roles": {n: role for n, (role, _) in models.items()},
    "cold_jd": cold_jd, "cold_resume": cold_res,
    "paired_auc_gaps": gaps,
    "error_analysis_cold_resume": errors,
    "jobswipe_match_badge_by_label": badge,
    "most_confident_errors_cold_resume": worst,
}
OUT.mkdir(exist_ok=True)
(OUT / "evaluation.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
np.savez_compressed(OUT / "oof_scores.npz", y=y, is_test=is_test,
                    **{f"{n}_score": oof[n][0] for n in oof},
                    **{f"{n}_pred": oof[n][1] for n in oof})
print(f"\nSaved results/evaluation.json ({report['duration_seconds']}s)")

## 11. Results tables

In [ ]:
def print_report(rep: dict) -> None:
    for split in ("cold_jd", "cold_resume"):
        print(f"\n{split.upper()}")
        print(f"{'model':<22}{'AUC':>7}{'95% CI':>17}{'macroF1':>9}{'NDCGseek':>10}"
              f"{'NDCGemp':>9}{'NoFit@5':>9}{'recGood':>9}")
        for n, r in rep[split].items():
            ci = f"[{r['auc_ci'][0]:.3f},{r['auc_ci'][1]:.3f}]"
            print(f"{n:<22}{r['auc_good_vs_no']:>7.3f}{ci:>17}{r['macro_f1']:>9.3f}"
                  f"{r['ndcg_seeker_at_10']:>10.3f}{r['ndcg_employer_at_10']:>9.3f}"
                  f"{r['nofit_share_top5_seeker']:>9.3f}{r['recall_per_class']['Good Fit']:>9.3f}")
    print("\nPAIRED AUC GAPS (95% cluster-bootstrap CI)")
    for k, v in rep["paired_auc_gaps"].items():
        print(f"  {k:<46} cold_jd {v['cold_jd']['point']:+.3f} {v['cold_jd']['ci']}  "
              f"cold_resume {v['cold_resume']['point']:+.3f} {v['cold_resume']['ci']}")
    print("\nERROR ANALYSIS (cold-resume OOF) AUC / severe-error rate")
    for cond, rows in rep["error_analysis_cold_resume"].items():
        print(f"  {cond}")
        for row in rows:
            cells = "  ".join(f"{n[:14]} {row[n]['auc']:.3f}/{row[n]['severe_error_rate']:.3f}"
                              for n in SHOWN)
            print(f"    {row['group']:<28}{row['n']:>6}  {cells}")
    print("\nJobSwipe badge by label:", json.dumps(rep["jobswipe_match_badge_by_label"]))


print_report(report)

In [ ]:
# The same headline numbers as DataFrames
cols = ["auc_good_vs_no", "auc_ci", "macro_f1", "ndcg_seeker_at_10", "ndcg_employer_at_10", "nofit_share_top5_seeker"]
for split in ("cold_resume", "cold_jd"):
    print(split)
    display(pd.DataFrame({n: {c: r[c] for c in cols} for n, r in report[split].items()}).T)

## 12. Figures
The figures are saved to `results/figures/` and also shown inline.

In [ ]:
import matplotlib

import matplotlib.pyplot as plt
import numpy as np

RES = ROOT / "results"
FIG = RES / "figures"

INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8985", "#e4e3df", "#fcfcfb"
# Color follows the model everywhere. Baselines are neutral; candidates take slots 1-2.
COLOR = {"random": "#c3c2b7", "tfidf_cosine": INK2,
         "jobswipe_gte_cosine": "#2a78d6", "jobtinder_lightgbm": "#eb6834",
         "jobswipe_resume_only": "#7fb0ea", "bge_cosine": "#f2a27f"}
LABEL = {"random": "Random (floor)", "tfidf_cosine": "Baseline: TF-IDF cosine",
         "jobswipe_gte_cosine": "JobSwipe (gte-small cosine)",
         "jobtinder_lightgbm": "Job-Tinder (LightGBM)",
         "jobswipe_resume_only": "  JobSwipe, resume only", "bge_cosine": "  bge-small cosine only"}
ORDER = ["random", "tfidf_cosine", "jobswipe_gte_cosine", "jobtinder_lightgbm",
         "jobswipe_resume_only", "bge_cosine"]

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 10, "axes.titleweight": "bold",
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.color": GRID, "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.fontsize": 8,
})


def fig_comparison(rep: dict) -> None:
    """AUC (with 95% CI), NDCG@10 seeker and macro F1, both splits, every model."""
    metrics = [("auc_good_vs_no", "AUC, Good vs No Fit"), ("ndcg_seeker_at_10", "NDCG@10, seeker deck"),
               ("macro_f1", "Macro F1, 3 classes")]
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.3), sharey=True)
    ys = np.arange(len(ORDER))[::-1]
    for ax, (key, title) in zip(axes, metrics):
        for y, name in zip(ys, ORDER):
            c = COLOR[name]
            for split, marker, fill, dy in (("cold_jd", "o", "none", 0.15), ("cold_resume", "o", c, -0.15)):
                r = rep[split][name]
                v = r[key]
                if key == "auc_good_vs_no":
                    lo, hi = r["auc_ci"]
                    ax.plot([lo, hi], [y + dy] * 2, color=c, lw=2, solid_capstyle="round", alpha=0.55)
                ax.plot(v, y + dy, marker=marker, ms=7, mfc=fill, mec=c, mew=2, ls="none")
        ax.set_title(title, loc="left")
        ax.tick_params(axis="y", length=0)
        if key == "auc_good_vs_no":
            ax.axvline(0.5, color=MUTED, lw=1, ls=":")
    axes[0].set_yticks(ys, [LABEL[n] for n in ORDER])
    for t, n in zip(axes[0].get_yticklabels(), ORDER):
        t.set_color(INK if n in ("tfidf_cosine", "jobswipe_gte_cosine", "jobtinder_lightgbm") else INK2)
    h = [plt.Line2D([], [], marker="o", ms=7, mfc="none", mec=INK2, mew=2, ls="none"),
         plt.Line2D([], [], marker="o", ms=7, mfc=INK2, mec=INK2, mew=2, ls="none")]
    fig.legend(h, ["Cold-JD split (unseen jobs, n=1,759)", "Cold-resume split (unseen resumes, 5-fold, n=8,000)"],
               loc="lower center", ncol=2, bbox_to_anchor=(0.55, -0.02))
    fig.text(0.01, 0.99, "Same data, same splits, same metrics. Lines on AUC are 95% cluster-bootstrap CIs.",
             color=INK2, va="top", fontsize=8)
    fig.tight_layout(rect=(0, 0.07, 1, 0.95))
    fig.savefig(FIG / "fig1_model_comparison.png", dpi=200)
    plt.close(fig)


def fig_errors(rep: dict) -> None:
    """AUC by group for the conditions that separate the models."""
    shown = ["tfidf_cosine", "jobswipe_gte_cosine", "jobtinder_lightgbm"]
    conds = [("jd_length", "Job description length"), ("resume_title_detected", "JobSwipe finds a job title"),
             ("skill_coverage", "Resume covers job's skills"), ("job_seniority", "Job seniority")]
    ea = rep["error_analysis_cold_resume"]
    fig, axes = plt.subplots(1, len(conds), figsize=(11, 3.2))
    for ax, (cond, title) in zip(axes, conds):
        rows = [r for r in ea[cond] if r["n"] >= 150]
        ys = np.arange(len(rows))[::-1]
        for off, name in zip((0.18, 0.0, -0.18), shown):  # dodge so equal values stay visible
            vals = [r[name]["auc"] for r in rows]
            ax.plot(vals, ys + off, marker="o", ms=7, ls="none", color=COLOR[name], mec=SURFACE, mew=1.2,
                    label=LABEL[name])
        ax.set_yticks(ys, [f"{r['group']}\n(n={r['n']:,})" for r in rows], fontsize=7.5)
        ax.tick_params(axis="y", length=0)
        ax.axvline(0.5, color=MUTED, lw=1, ls=":")
        ax.set_title(title, loc="left", fontsize=9)
        ax.set_xlim(0.4, 0.9)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3)
    fig.text(0.01, 0.99, "AUC (Good vs No Fit) within each group, cold-resume out-of-fold predictions. "
             "Groups with n < 150 omitted.", color=INK2, va="top", fontsize=8)
    fig.tight_layout(rect=(0, 0.08, 1, 0.95))
    fig.savefig(FIG / "fig2_error_analysis.png", dpi=200)
    plt.close(fig)


def fig_badge() -> None:
    """JobSwipe's displayed '% match' for pairs of each true label."""
    d = np.load(RES / "oof_scores.npz")
    y, cos = d["y"], d["jobswipe_gte_cosine_score"]
    pct = np.clip((cos - 0.80) / 0.16, 0, 1) * 100
    fig, ax = plt.subplots(figsize=(6.2, 2.6))
    bins = np.linspace(0, 100, 21)
    shades = {0: "#b8d3f5", 1: "#6aa0e6", 2: "#1c4f8f"}  # one hue, light -> dark (ordinal)
    for lab, name in ((0, "No Fit"), (1, "Potential Fit"), (2, "Good Fit")):
        h, _ = np.histogram(pct[y == lab], bins=bins)
        ax.plot((bins[:-1] + bins[1:]) / 2, h / h.sum() * 100, color=shades[lab], lw=2,
                label=f"{name} (median {np.median(pct[y == lab]):.0f}%)")
    ax.set_xlabel("'% match' badge JobSwipe would display")
    ax.set_ylabel("% of pairs")
    ax.grid(axis="y", color=GRID)
    ax.legend(loc="upper left")
    ax.set_title("Displayed match % overlaps heavily across true labels", loc="left")
    fig.tight_layout()
    fig.savefig(FIG / "fig3_jobswipe_badge.png", dpi=200)
    plt.close(fig)

In [ ]:
from IPython.display import Image, display as show

FIG.mkdir(parents=True, exist_ok=True)
rep = json.loads((RES / "evaluation.json").read_text(encoding="utf-8"))
fig_comparison(rep)
fig_errors(rep)
fig_badge()
for name in ("fig1_model_comparison.png", "fig2_error_analysis.png", "fig3_jobswipe_badge.png"):
    show(Image(filename=str(FIG / name)))

## 13. Test 2: neutral-ground check
The labelled data above is Job-Tinder's training source, so this check uses data **neither** model was trained on: 240 Kaggle resumes (10 per category) and real LinkedIn postings, both from the D1 project folder (`../dtsc3602_proj`).

Each resume gets the same kind of candidate list: 10 postings in its field (the title matches the resume's category) and 40 that are not. Each model ranks those 50.
- JobSwipe runs as in production, with real titles and locations.
- Job-Tinder uses its **shipped** model files and is not retrained.
- The metrics are per-resume AUC, precision@10, NDCG@10 and the seniority-mismatch rate in the top 10, with bootstrap CIs over resumes. Results are written to `results/in_domain_check.json`.

About 15 minutes the first time, mostly embedding.

In [ ]:
import json
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()  # run this notebook from the repo root
JT = ROOT / "repos" / "job_tinder_test"
sys.path.insert(0, str(JT))
D1 = ROOT.parent / "dtsc3602_proj"
CACHE = ROOT / "eval" / "cache"
OUT = ROOT / "results"

import joblib  # noqa: E402
import lightgbm as lgb  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.metrics import ndcg_score, roc_auc_score  # noqa: E402
from sklearn.preprocessing import normalize  # noqa: E402

from ml.encoder import get_encoder, job_text  # noqa: E402
from ml.evaluate_in_domain import CATEGORY_TITLES  # noqa: E402
from ml.features import JobSide, ResumeSide, build_matrix, fit_score, resume_years  # noqa: E402

SEED = 42
PER_CATEGORY = 10
N_ON, N_OFF = 10, 40
POOL_ON, POOL_RANDOM = 25, 600
TOP = 10
BOOT = 1000
MODELS = ["random", "tfidf_cosine", "jobswipe_gte_cosine", "jobtinder_lightgbm"]


def load_sample():
    rng = np.random.default_rng(SEED)
    res = pd.read_csv(D1 / "resumes" / "Resume" / "Resume.csv", usecols=["ID", "Resume_str", "Category"])
    res = res.groupby("Category").sample(PER_CATEGORY, random_state=SEED).reset_index(drop=True)
    res["Resume_str"] = res.Resume_str.str.strip()

    cols = ["job_id", "title", "company_name", "description", "location", "remote_allowed",
            "formatted_experience_level"]
    jobs = pd.read_csv(D1 / "postings" / "postings.csv", usecols=cols)
    jobs = jobs[jobs.title.notna() & (jobs.description.fillna("").str.len() >= 700)]
    jobs = jobs.drop_duplicates(["company_name", "title"]).reset_index(drop=True)
    pats = {c: re.compile(p) for c, p in CATEGORY_TITLES.items()}
    low = jobs.title.str.lower()
    match = pd.DataFrame({c: low.str.contains(p.pattern.replace("(", "(?:")) for c, p in pats.items()})

    # Shared pool (~1,200 postings): 25 on-target per category + 600 random postings.
    pool = set()
    for c in CATEGORY_TITLES:
        pool |= set(rng.choice(np.flatnonzero(match[c].to_numpy()), POOL_ON, replace=False))
    pool |= set(rng.choice(len(jobs), POOL_RANDOM, replace=False))
    pool = np.array(sorted(pool))
    in_pool = np.zeros(len(jobs), bool)
    in_pool[pool] = True

    # Candidate lists: same recipe for every resume, fixed by the seed. Off-target postings
    # come from the whole pool, so some are other categories' on-target jobs (harder negatives).
    rows = []
    for i, r in res.iterrows():
        m = match[r.Category].to_numpy()
        on = rng.choice(np.flatnonzero(m & in_pool), N_ON, replace=False)
        off = rng.choice(np.flatnonzero(~m & in_pool), N_OFF, replace=False)
        rows += [(i, j, 1) for j in on] + [(i, j, 0) for j in off]
    pairs = pd.DataFrame(rows, columns=["r", "j", "on_target"])
    used = np.sort(pairs.j.unique())
    jobs = jobs.iloc[used].reset_index(drop=True)
    pairs["j"] = pairs.j.map({old: new for new, old in enumerate(used)})
    return res, jobs, pairs


def jobswipe_vectors(res, jobs):
    key = CACHE / "in_domain_jobswipe.npz"
    if key.exists():
        z = np.load(key)
        return z["R"], z["J"]
    src, dst = CACHE / "in_domain_js_in.json", CACHE / "in_domain_js_out.json"
    src.write_text(json.dumps({
        "resumes": res.Resume_str.tolist(),
        "jobs": [{"title": t, "description": d, "location": l if isinstance(l, str) else None,
                  "is_remote": bool(rm == 1.0)}
                 for t, d, l, rm in zip(jobs.title, jobs.description, jobs.location, jobs.remote_allowed)],
    }))
    subprocess.run(["node", str(ROOT / "eval" / "jobswipe_texts.mts"), str(src), str(dst)], check=True, cwd=ROOT)
    texts = json.loads(dst.read_text(encoding="utf-8"))
    profiles = texts["profiles_suggested"]
    print(f"  JobSwipe found a title for {sum(p['header'].startswith('Role:') for p in profiles)}"
          f"/{len(profiles)} resumes")

    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer("thenlper/gte-small", device="cpu")
    m.max_seq_length = 512
    flat = sorted({t for p in profiles for t in [p["header"], *p["chunks"]] if t} | set(texts["jobs"]))
    print(f"  embedding {len(flat):,} strings with gte-small ...", flush=True)
    vec = dict(zip(flat, m.encode(flat, batch_size=16, normalize_embeddings=True, show_progress_bar=True)))

    def prof(p):
        h, ch = p["header"], p["chunks"]
        parts = ([(h, .5)] + [(c, .5 / len(ch)) for c in ch]) if h and ch else [(h, 1.)] if h else \
                [(c, 1 / len(ch)) for c in ch]
        v = sum(w * vec[t] for t, w in parts)
        return v / np.linalg.norm(v)
    R = np.stack([prof(p) for p in profiles])
    J = np.stack([vec[t] for t in texts["jobs"]])
    np.savez_compressed(key, R=R, J=J)
    return R, J


def jobtinder_scores(res, jobs, pairs):
    key = CACHE / "in_domain_jobtinder_vecs.npz"
    jt_texts = [job_text(t, c if isinstance(c, str) else None, d[:4000])
                for t, c, d in zip(jobs.title, jobs.company_name, jobs.description)]
    if key.exists():
        z = np.load(key)
        R, J = z["R"], z["J"]
    else:
        enc = get_encoder()
        print(f"  embedding {len(res)} resumes + {len(jobs):,} postings with bge-small ...", flush=True)
        R, J = enc.encode_resumes(res.Resume_str.tolist()), enc.encode_jobs(jt_texts)
        np.savez_compressed(key, R=R, J=J)
    art = joblib.load(JT / "models" / "fit_artifacts.joblib")
    booster = lgb.Booster(model_file=str(JT / "models" / "fit_model.txt"))
    rt = [res.Resume_str[i] for i in pairs.r]
    jt = [jt_texts[j] for j in pairs.j]
    tf = np.asarray(normalize(art["vectorizer"].transform(rt)).multiply(
        normalize(art["vectorizer"].transform(jt))).sum(axis=1)).ravel()
    X = build_matrix([ResumeSide(text=t, embedding=R[i]) for t, i in zip(rt, pairs.r)],
                     [JobSide(text=t, embedding=J[j]) for t, j in zip(jt, pairs.j)],
                     tfidf_cosines=tf, skill_idf=art["skill_idf"])
    from ml.features import ALL_FEATURE_NAMES
    cols = [ALL_FEATURE_NAMES.index(n) for n in art["feature_names"]]
    return fit_score(booster.predict(X[:, cols]))


def per_resume(pairs, score, mismatch):
    out = []
    for _, g in pairs.assign(s=score, mm=mismatch).groupby("r"):
        top = g.nlargest(TOP, "s")
        out.append((roc_auc_score(g.on_target, g.s), top.on_target.mean(),
                    ndcg_score([g.on_target.to_numpy()], [g.s.to_numpy()], k=TOP), top.mm.mean()))
    return np.array(out)


def main():
    t0 = time.time()
    print("Sampling resumes and postings ...")
    res, jobs, pairs = load_sample()
    print(f"  {len(res)} resumes x {N_ON + N_OFF} candidates = {len(pairs):,} pairs, "
          f"{len(jobs):,} unique postings")

    scores = {"random": np.random.default_rng(SEED).random(len(pairs))}
    tv = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit(
        pd.concat([res.Resume_str, jobs.title + " " + jobs.description]))
    Rt = normalize(tv.transform(res.Resume_str))
    Jt = normalize(tv.transform(jobs.title + " " + jobs.description))
    scores["tfidf_cosine"] = np.asarray(Rt[pairs.r.to_numpy()].multiply(Jt[pairs.j.to_numpy()]).sum(axis=1)).ravel()
    print("JobSwipe ...")
    R, J = jobswipe_vectors(res, jobs)
    scores["jobswipe_gte_cosine"] = np.einsum("ij,ij->i", R[pairs.r], J[pairs.j])
    print("Job-Tinder (shipped model) ...")
    scores["jobtinder_lightgbm"] = jobtinder_scores(res, jobs, pairs)

    # seniority mismatch flag per pair
    yrs = np.array([resume_years(t) for t in res.Resume_str])
    lvl = jobs.formatted_experience_level.fillna("").to_numpy()
    ry, jl = yrs[pairs.r], lvl[pairs.j]
    mismatch = (((ry <= 2) & np.isin(jl, ["Director", "Executive"])) |
                ((ry >= 10) & np.isin(jl, ["Internship", "Entry level"]))).astype(float)

    rng = np.random.default_rng(SEED)
    boot = rng.integers(0, len(res), (BOOT, len(res)))
    per = {m: per_resume(pairs, scores[m], mismatch) for m in MODELS}
    names = ["auc", "precision_at_10", "ndcg_at_10", "seniority_mismatch_at_10"]
    report = {"n_resumes": len(res), "n_pairs": int(len(pairs)), "n_postings": int(len(jobs)),
              "candidates_per_resume": {"on_target": N_ON, "off_target": N_OFF}, "models": {}, "gaps_vs_baseline": {},
              "auc_by_category": {}}
    for m in MODELS:
        report["models"][m] = {n: {"mean": round(float(per[m][:, k].mean()), 4),
                                   "ci": [round(float(x), 4) for x in np.percentile(per[m][boot, k].mean(1), [2.5, 97.5])]}
                               for k, n in enumerate(names)}
    for m in ("jobswipe_gte_cosine", "jobtinder_lightgbm"):
        report["gaps_vs_baseline"][m] = {n: {"mean": round(float((per[m][:, k] - per["tfidf_cosine"][:, k]).mean()), 4),
                                             "ci": [round(float(x), 4) for x in np.percentile(
                                                 (per[m][boot, k] - per["tfidf_cosine"][boot, k]).mean(1), [2.5, 97.5])]}
                                         for k, n in enumerate(names)}
    d = per["jobtinder_lightgbm"] - per["jobswipe_gte_cosine"]
    report["jobtinder_minus_jobswipe"] = {n: {"mean": round(float(d[:, k].mean()), 4),
                                              "ci": [round(float(x), 4) for x in np.percentile(d[boot, k].mean(1), [2.5, 97.5])]}
                                          for k, n in enumerate(names)}
    for cat, idx in res.groupby("Category").groups.items():
        report["auc_by_category"][cat] = {m: round(float(per[m][list(idx), 0].mean()), 3) for m in MODELS}
    report["duration_seconds"] = round(time.time() - t0, 1)
    OUT.mkdir(exist_ok=True)
    (OUT / "in_domain_check.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

    print(f"\n{'model':<22}" + "".join(f"{n:>30}" for n in names))
    for m in MODELS:
        print(f"{m:<22}" + "".join(f"{report['models'][m][n]['mean']:>10.3f} [{report['models'][m][n]['ci'][0]:.3f},{report['models'][m][n]['ci'][1]:.3f}]"
                                   for n in names))
    for k, v in {**report["gaps_vs_baseline"], "jobtinder - jobswipe": report["jobtinder_minus_jobswipe"]}.items():
        print(f"gap {k:<26}" + "".join(f"{v[n]['mean']:>+10.3f} [{v[n]['ci'][0]:+.3f},{v[n]['ci'][1]:+.3f}]" for n in names))
    print("\nAUC by category:")
    for cat, v in report["auc_by_category"].items():
        print(f"  {cat:<24}" + "  ".join(f"{m[:10]} {v[m]:.3f}" for m in MODELS[1:]))
    print(f"\nSaved results/in_domain_check.json ({report['duration_seconds']}s)")

In [ ]:
main()